# 查看历史

In [6]:
from typing import TypedDict
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import MemorySaver

class HistoryState(TypedDict):
    count: int
    history: list

def increment_node(state: HistoryState) -> dict:
    new_count = state.get("count", 0) + 1
    return {
        "count": new_count,
        "history": state.get("history", []) + [f"Incremented to {new_count}"]
    }

graph = StateGraph(HistoryState)
graph.add_node("inc1", increment_node)
graph.add_node("inc2", increment_node)
graph.add_node("inc3", increment_node)

graph.add_edge(START, "inc1")
graph.add_edge("inc1", "inc2")
graph.add_edge("inc2", "inc3")
graph.add_edge("inc3", END)

app = graph.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "history-demo"}}

# 运行
result = app.invoke({"count": 0, "history": []}, config)
print(f"最终结果: {result}")

# 查看历史
print("\n=== 执行历史 ===")
history = app.get_state_history(config)
for i, checkpoint in enumerate(history):
    print(f"\nCheckpoint {i + 1}:")
    print(f"  State: {checkpoint.values}")
    print(f"  Next: {checkpoint.next}")
    print(f"  Checkpoint ID: {checkpoint.config['configurable'].get('checkpoint_id')}")

最终结果: {'count': 3, 'history': ['Incremented to 1', 'Incremented to 2', 'Incremented to 3']}

=== 执行历史 ===

Checkpoint 1:
  State: {'count': 3, 'history': ['Incremented to 1', 'Incremented to 2', 'Incremented to 3']}
  Next: ()
  Checkpoint ID: 1f16fc4b-92df-6b1e-8003-b769dca32c9f

Checkpoint 2:
  State: {'count': 2, 'history': ['Incremented to 1', 'Incremented to 2']}
  Next: ('inc3',)
  Checkpoint ID: 1f16fc4b-92df-6178-8002-9079818b07c8

Checkpoint 3:
  State: {'count': 1, 'history': ['Incremented to 1']}
  Next: ('inc2',)
  Checkpoint ID: 1f16fc4b-92dd-69cc-8001-2fcff6c5dddd

Checkpoint 4:
  State: {'count': 0, 'history': []}
  Next: ('inc1',)
  Checkpoint ID: 1f16fc4b-92da-6a7e-8000-fad0345d3a4a

Checkpoint 5:
  State: {}
  Next: ('__start__',)
  Checkpoint ID: 1f16fc4b-92d9-658e-bfff-f75eaf65f591


# 回溯到历史状态

In [10]:
# 获取历史中的某个 checkpoint
history_list = list(app.get_state_history(config))

# 选择第 2 个 checkpoint（count = 1）
target_checkpoint = history_list[-3]  # 倒数第三个

print(f"\n回溯到: {target_checkpoint.values}")

# 从这个 checkpoint 继续执行（会创建新分支）
new_config = {
    "configurable": {
        "thread_id": "history-demo",
        "checkpoint_id": target_checkpoint.config["configurable"]["checkpoint_id"]
    }
}
# 继续执行
result = app.invoke(None, new_config)
print(f"从历史继续的结果: {result}")


回溯到: {'count': 1, 'history': ['Incremented to 1']}
从历史继续的结果: {'count': 3, 'history': ['Incremented to 1', 'Incremented to 2', 'Incremented to 3']}
